In [1]:
import torch
import torchvision
import torch.nn.functional as F
from pathlib import Path
import wandb
import shutil
import os
import random
import numpy
import PIL

Creiamo la struttura della repository

In [2]:
random.seed(1)

gallery_folder = "gallery"
query_folder = "query"
celeb = "img_align_celeba" #questo è il nostro dataset di training

os.makedirs(gallery_folder, exist_ok = True)
os.makedirs(query_folder, exist_ok = True)


random.shuffle(os.listdir(celeb)) #we shaffle the images to randomize the selection
len(os.listdir(celeb))

202388

Otteniamo un sample randomico di 200 immagini dalle 200k per un test più rapido

In [ ]:
# Imposta il seed per la riproducibilità
random.seed(1)

# Prendiamo 100 immagini casuali per la gallery e per la query
gallery_images_sample = random.sample(os.listdir(celeb), 100)
query_images_sample = random.sample(os.listdir(celeb), 100)

# Sposta ogni file dalla lista
for q_image, g_image in zip(query_images_sample, gallery_images_sample):
    source_file_q = os.path.join(celeb, q_image)  # Percorso completo del file sorgente per la query
    source_file_g = os.path.join(celeb, g_image)  # Percorso completo del file sorgente per la gallery
    
    destination_file_q = os.path.join(query_folder, q_image)  # Percorso completo per la destinazione della query
    destination_file_g = os.path.join(gallery_folder, g_image)  # Percorso completo per la destinazione della gallery

    # Controlla se il file esiste già nella cartella di destinazione
    if not os.path.exists(destination_file_q):  # Se il file non esiste nella query_folder
        shutil.move(source_file_q, destination_file_q)
        # print(f"Spostato: {q_image} in {query_folder}")
    # else:
        # print(f"Immagine {q_image} già presente in {query_folder}, salto lo spostamento.")
    
    if not os.path.exists(destination_file_g):  # Se il file non esiste nella gallery_folder
        shutil.move(source_file_g, destination_file_g)
        # print(f"Spostato: {g_image} in {gallery_folder}")
    # else:
        # print(f"Immagine {g_image} già presente in {gallery_folder}, salto lo spostamento.")

IndentationError: expected an indented block after 'else' statement on line 20 (2668424871.py, line 23)

In [ ]:
print(len(os.listdir(query_folder)))
print(len(os.listdir(gallery_folder)))


# print(len(numpy.unique(os.listdir(query_folder))))

200


passiamo ora all'ingestion vera e propria. Carichiamo le immagini in python come tensors RGB  

facciamo un breve test per verificare che tutto funzioni

In [15]:
from torchvision import transforms
from PIL import Image

#piccolo test per verificare che funzioni tutto
transform = transforms.Compose([
    transforms.ToTensor(),  # Converte l'immagine in un tensor [0, 1]
])

# Carica un'immagine (assicurati che sia in formato RGB)
my_path = query_folder + "/" + os.listdir(query_folder)[1]
image = Image.open(my_path).convert("RGB")

# Applica la trasformazione
image_tensor = transform(image)

# Visualizza la forma del tensor
print(image_tensor.shape)  # Dovrebbe essere (3, H, W)

torch.Size([3, 218, 178])


Costruiamo una classe ImageRetrievalDataset per caricare ed elaborare in tensori le immagini

In [23]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

class ImageRetrievalDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_names = os.listdir(image_dir)  # Ottieni i nomi dei file nella cartella

    def __len__(self):
        return len(self.image_names)  # Restituisce il numero di immagini

    def __getitem__(self, idx):
        image_name = self.image_names[idx]  # Prendi il nome dell'immagine
        image_path = os.path.join(self.image_dir, image_name)  # Costruisci il percorso completo
        image = Image.open(image_path).convert("RGB")  # Carica l'immagine e convertila in RGB

        if self.transform:
            image = self.transform(image)  # Applica le trasformazioni (es. ToTensor)

        return image, image_name  # Restituisce l'immagine e il suo nome (facoltativo)

# Trasformazioni da applicare (ad esempio, ToTensor per ottenere un tensor) oltre che la data augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Ridimensiona a una dimensione uniforme
    transforms.RandomHorizontalFlip(),  # Flip orizzontale casuale
    transforms.RandomRotation(15),  # Rotazione casuale tra -15 e 15 gradi
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # Modifiche di colore
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # Ritaglio casuale con dimensioni variabili
    transforms.ToTensor() ,  # Converte in un tensor RGB
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]) #normalizziamo i dati
])



Possiamo fare ingesting dei datasets in formato tensor RGB  

Facciamo l'ingestion della gallery

In [ ]:
# Inizializza il dataset con il percorso della cartella delle immagini
gallery_dataset = ImageRetrievalDataset(image_dir=gallery_folder, transform=transform)

# Crea un DataLoader per caricare le immagini in batch
data_loader_gal = torch.utils.data.DataLoader(gallery_dataset, batch_size=32, shuffle=True, num_workers = 2)

# Itera sui dati per vedere la forma del tensor
for images, image_names in data_loader_gal:
    print(images.shape)  # [batch_size, 3, 224, 224]
    break

torch.Size([32, 3, 224, 224])


Facciamo l'ingestion della query

In [ ]:
# Inizializza il dataset con il percorso della cartella delle immagini
query_dataset = ImageRetrievalDataset(image_dir=query_folder, transform=transform)

# Crea un DataLoader per caricare le immagini in batch
data_loader_que = torch.utils.data.DataLoader(query_dataset, batch_size=32, shuffle=True, num_workers= 2)

# Itera sui dati per vedere la forma del tensor
for images, image_names in data_loader_que:
    print(images.shape)  # [batch_size, 3, 224, 224]
    break

torch.Size([32, 3, 224, 224])
